In [ ]:

from google.colab import auth
from google.cloud import bigquery

# Autenticar com a conta Google
auth.authenticate_user()

# Inicializar o cliente BigQuery
PROJECT_ID = "macrodata-brasil"

client = bigquery.Client(
    project=PROJECT_ID,
    location="southamerica-east1"
)

print("Autenticação realizada com sucesso!")


Autenticação realizada com sucesso!


In [ ]:

import requests
import pandas as pd

from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from google.cloud import bigquery

# Configuração
PROJECT_ID = "macrodata-brasil"
TABELA_BRONZE = f"{PROJECT_ID}.bronze.bcb_selic_raw"

URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"

DATA_INICIAL_HISTORICA = datetime(2024, 1, 1).date()
JANELA_REPROCESSAMENTO = 30

# Cliente do BigQuery
client = bigquery.Client(project=PROJECT_ID)

# 1. Consultar última data disponível na Bronze
consulta = f"""
SELECT
    MAX(SAFE.PARSE_DATE('%d/%m/%Y', data)) AS ultima_data
FROM `{TABELA_BRONZE}`
"""

resultado = client.query(consulta).result()
ultima_data = next(iter(resultado)).ultima_data

# 2. Definir período incremental
hoje = datetime.now(
    ZoneInfo("America/Sao_Paulo")
).date()

if ultima_data is None:
    data_inicial = DATA_INICIAL_HISTORICA
else:
    data_inicial = max(
        DATA_INICIAL_HISTORICA,
        ultima_data - timedelta(
            days=JANELA_REPROCESSAMENTO
        )
    )

print(f"Última data na Bronze: {ultima_data}")
print(f"Início da extração: {data_inicial}")
print(f"Fim da extração: {hoje}")

# 3. Requisitar dados da API
parametros = {
    "formato": "json",
    "dataInicial": data_inicial.strftime("%d/%m/%Y"),
    "dataFinal": hoje.strftime("%d/%m/%Y")
}

resposta = requests.get(
    URL,
    params=parametros,
    timeout=30
)

resposta.raise_for_status()

# 4. Converter resposta
df_selic = pd.DataFrame(resposta.json())

# 5. Adicionar metadados
df_selic["data_extracao"] = datetime.now(timezone.utc)
df_selic["fonte"] = "BCB_SGS_432"

print(f"Registros extraídos: {len(df_selic)}")

display(df_selic.head())


Última data na Bronze: 2026-10-31
Início da extração: 2026-10-01
Fim da extração: 2026-11-17
Registros extraídos: 48


,data,valor,data_extracao,fonte
0,01/10/2026,13.75,2026-11-17 11:56:57.258185+00:00,BCB_SGS_432
1,02/10/2026,13.75,2026-11-17 11:56:57.258185+00:00,BCB_SGS_432
2,03/10/2026,13.75,2026-11-17 11:56:57.258185+00:00,BCB_SGS_432
3,04/10/2026,13.75,2026-11-17 11:56:57.258185+00:00,BCB_SGS_432
4,05/10/2026,13.75,2026-11-17 11:56:57.258185+00:00,BCB_SGS_432


In [ ]:

import io
import json
import pandas as pd
from google.cloud import bigquery

# Tabela intermediária
TABELA_STAGING = (
    "macrodata-brasil.bronze.bcb_selic_staging"
)

# 1. Validar os dados recebidos
if df_selic.empty:
    raise ValueError(
        "Extração vazia. Carregamento cancelado."
    )

registros = df_selic[
    ["data", "valor", "data_extracao", "fonte"]
].copy()

# 2. Validar datas e valores
datas = pd.to_datetime(
    registros["data"],
    format="%d/%m/%Y",
    errors="coerce"
)

valores = pd.to_numeric(
    registros["valor"],
    errors="coerce"
)

if datas.isna().any() or valores.isna().any():
    raise ValueError(
        "Existem datas ou valores inválidos."
    )

if datas.duplicated().any():
    raise ValueError(
        "Existem datas duplicadas na extração."
    )

# 3. Formatar o timestamp
registros["data_extracao"] = (
    pd.to_datetime(
        registros["data_extracao"],
        utc=True
    ).dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
)

# 4. Converter para JSONL
conteudo_jsonl = "\n".join(
    json.dumps(registro, ensure_ascii=False)
    for registro in registros.to_dict(orient="records")
)

arquivo_memoria = io.BytesIO(
    conteudo_jsonl.encode("utf-8")
)

# 5. Configuração da tabela de staging
configuracao = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("data", "STRING"),
        bigquery.SchemaField("valor", "STRING"),
        bigquery.SchemaField("data_extracao", "TIMESTAMP"),
        bigquery.SchemaField("fonte", "STRING"),
    ],
)

# 6. Carregar somente na staging
job = client.load_table_from_file(
    arquivo_memoria,
    TABELA_STAGING,
    job_config=configuracao,
    location="southamerica-east1",
)

job.result()

# 7. Validar quantidade carregada
tabela = client.get_table(TABELA_STAGING)

if tabela.num_rows != len(registros):
    raise ValueError(
        "Quantidade carregada diferente da extração."
    )

print("Staging carregada com sucesso!")
print("Registros carregados:", tabela.num_rows)
print("Tabela:", TABELA_STAGING)
print("ID do job:", job.job_id)


Staging carregada com sucesso!
Registros carregados: 48
Tabela: macrodata-brasil.bronze.bcb_selic_staging
ID do job: cc2ae582-a118-422c-a4e7-53f30f4815b8
